# Phase 3 — Duplicate Audit & Fixed Splits

**Objective:** Detect duplicate relationships among BUSI images and create one
immutable five-fold grouped split for every later experiment.

**Rules:**
- Detect exact duplicates using SHA-256.
- Detect near-duplicate candidates using perceptual hashes.
- Verify near-duplicate candidates with SSIM.
- Assign stable group IDs.
- Create group-disjoint, stratified five-fold split.
- No BUS-UCLM sample appears in any split.
- Split is immutable — any future correction creates v2 and a deviation.

**No model training in this phase.**

**Status labels:** `planned` | `implemented` | `runnable` | `executed` | `validated` | `failed` | `blocked`

## 3.0 — Colab bootstrap

Detects Google Colab and clones the repository if needed. In VS Code, does nothing.

In [1]:
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
COLAB_TARGET = Path("/content/CausalMask-XAI")

if is_colab():
    print("Detected Google Colab environment.")
    if COLAB_TARGET.exists() and (COLAB_TARGET / "CausalMask-XAI.md").exists():
        print(f"Repository present at {COLAB_TARGET}. Pulling latest...")
        !cd {COLAB_TARGET} && git pull --ff-only
        print("Repository updated to latest commit.")
    else:
        if COLAB_TARGET.exists():
            import shutil
            shutil.rmtree(COLAB_TARGET)
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {COLAB_TARGET}
        assert (COLAB_TARGET / "CausalMask-XAI.md").exists(), "Clone failed: marker file missing"
    os.environ["CAUSALMASK_PROJECT_ROOT"] = str(COLAB_TARGET)
    print(f"Set CAUSALMASK_PROJECT_ROOT={COLAB_TARGET}")
else:
    print("Not in Colab — skipping bootstrap.")

Detected Google Colab environment.
Repository present at /content/CausalMask-XAI. Pulling latest...
Updating ce4654d..02f68e7
error: Your local changes to the following files would be overwritten by merge:
	artifacts/phases/phase_02_status.json
	reports/data_audit.md
Please commit your changes or stash them before you merge.
Aborting
Repository updated to latest commit.
Set CAUSALMASK_PROJECT_ROOT=/content/CausalMask-XAI


## 3.1 — Resolve project root

Resolution order:
1. `CAUSALMASK_PROJECT_ROOT` environment variable
2. Walk up from cwd looking for `CausalMask-XAI.md`
3. Check `/content/CausalMask-XAI` (Colab default)

In [3]:
import sys
from pathlib import Path


def resolve_project_root() -> Path:
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT")
    if env_root:
        p = Path(env_root)
        if (p / "CausalMask-XAI.md").exists():
            return p.resolve()
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "CausalMask-XAI.md").exists():
            return candidate.resolve()
    colab_fallback = Path("/content/CausalMask-XAI")
    if colab_fallback.exists() and (colab_fallback / "CausalMask-XAI.md").exists():
        return colab_fallback.resolve()
    raise RuntimeError(
        "Cannot resolve project root. Set CAUSALMASK_PROJECT_ROOT or run from within the repo."
    )


PROJECT_ROOT = resolve_project_root()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
assert (PROJECT_ROOT / "CausalMask-XAI.md").exists(), "Marker file missing"

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

PROJECT_ROOT = /content/CausalMask-XAI


## 3.2 — Active configuration

In [4]:
import json
import platform
from datetime import datetime, timezone

import torch

CONFIG = {
    "project_root": str(PROJECT_ROOT),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "phase": "03",
    "phase_name": "Duplicate Audit and Fixed Splits",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "datasets": {
        "busi": {
            "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
            "extract_rel": "data/raw/extracted/busi",
            "manifest_rel": "data/manifests/busi_manifest_v1.parquet",
        },
        "bus_uclm": {
            "archive_rel": "data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip",
            "extract_rel": "data/raw/extracted/bus_uclm",
            "manifest_rel": "data/manifests/bus_uclm_manifest_v1.parquet",
        },
    },
    "seed": 42,
    "manifest_version": "v1",
    "grouped_manifest_version": "v2",
    "duplicate_version": "v1",
    "split_version": "v1",
    "split_name": "busi_binary_grouped_5fold_v1",
    "binary_classification": ["benign", "malignant"],
    "external_datasets": ["bus_uclm"],
    "phash_hash_size": 8,
    "phash_highfreq_factor": 4,
    "phash_min_similarity": 0.75,
    "phash_candidate_threshold": 0.75,
    "ssim_verification_threshold": 0.85,
    "n_splits": 5,
    "validation_fraction": 0.2,
}

print(json.dumps(CONFIG, indent=2, default=str))

{
  "project_root": "/content/CausalMask-XAI",
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cpu",
  "cuda_available": false,
  "cuda_version": null,
  "gpu_name": null,
  "phase": "03",
  "phase_name": "Duplicate Audit and Fixed Splits",
  "timestamp_utc": "2026-07-22T07:19:36.622132+00:00",
  "datasets": {
    "busi": {
      "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
      "extract_rel": "data/raw/extracted/busi",
      "manifest_rel": "data/manifests/busi_manifest_v1.parquet"
    },
    "bus_uclm": {
      "archive_rel": "data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip",
      "extract_rel": "data/raw/extracted/bus_uclm",
      "manifest_rel": "data/manifests/bus_uclm_manifest_v1.parquet"
    }
  },
  "seed": 42,
  "manifest_version": "v1",
  "grouped_manifest_version": "v2",
  "duplicate_version": "v1",
  "split_version": "v1",
  "split_name": "busi_binary_grouped_5fold_v1",
  "binary_classifi

## 3.3 — Deterministic seeds

In [5]:
import random
import numpy as np

SEED = CONFIG["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seeds set to {SEED}")
print(f"Python random: {random.random():.10f}")
print(f"NumPy: {np.random.rand():.10f}")
print(f"Torch: {torch.rand(1).item():.10f}")

Seeds set to 42
Python random: 0.6394267985
NumPy: 0.3745401188
Torch: 0.8822692633


## 3.4 — Create output directories, mount Drive, restore Phase 2 artifacts

Create local directories and mount Google Drive for persistent artifact storage.
Also restores manifests and extracted raw data from Drive if missing locally
(e.g. on a fresh Colab instance after git clone).

In [6]:
import shutil
import zipfile

MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"
PHASES_DIR = PROJECT_ROOT / "artifacts" / "phases"
ARCHIVES_DIR = PROJECT_ROOT / "data" / "raw" / "archives"
EXTRACT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"

for d in [MANIFESTS_DIR, SPLITS_DIR, REPORTS_DIR, PHASES_DIR, ARCHIVES_DIR, EXTRACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Manifests dir:  {MANIFESTS_DIR}")
print(f"Splits dir:     {SPLITS_DIR}")
print(f"Reports dir:    {REPORTS_DIR}")
print(f"Phases dir:     {PHASES_DIR}")

# Mount Google Drive for persistent storage across Colab sessions
DRIVE_BASE = None
if is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/CausalMask-XAI")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will sync to {DRIVE_BASE}")
    
    for subdir in ["manifests", "splits", "reports", "artifacts"]:
        (DRIVE_BASE / subdir).mkdir(parents=True, exist_ok=True)
else:
    print("Google Drive not mounted (not in Colab). Artifacts saved locally only.")


def save_to_drive(local_path: Path, subdir: str):
    if DRIVE_BASE is None:
        return
    target = DRIVE_BASE / subdir / local_path.name
    shutil.copy2(local_path, target)
    print(f"  -> Synced to Drive: {target}")


# ---- Restore Phase 2 artifacts from Drive (if missing locally) ----

def restore_from_drive(subdir: str, filename: str, local_dir: Path) -> bool:
    """Copy a single file from Drive to local if it doesn't exist locally."""
    if DRIVE_BASE is None:
        return False
    src = DRIVE_BASE / subdir / filename
    dst = local_dir / filename
    if dst.exists():
        return False
    if not src.exists():
        print(f"  [WARN] Not found on Drive either: {src}")
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Restored from Drive: {dst}")
    return True


print("\n--- Restoring Phase 2 artifacts from Drive ---")

# 1. Manifests (parquet + json)
for fname in [
    f"busi_manifest_{CONFIG['manifest_version']}.parquet",
    f"bus_uclm_manifest_{CONFIG['manifest_version']}.parquet",
    f"busi_manifest_summary_{CONFIG['manifest_version']}.json",
    f"bus_uclm_manifest_summary_{CONFIG['manifest_version']}.json",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

# 2. Archives — restore from Drive if missing locally (small ZIPs, avoids re-download)
for ds_name, cfg in CONFIG["datasets"].items():
    archive_path = PROJECT_ROOT / cfg["archive_rel"]
    restore_from_drive("archives", archive_path.name, ARCHIVES_DIR)

# 3. Extracted raw data — extract from archives (never stored on Drive)
for ds_name, cfg in CONFIG["datasets"].items():
    extract_path = PROJECT_ROOT / cfg["extract_rel"]
    if not extract_path.exists() or not any(extract_path.iterdir()):
        print(f"  {ds_name}: no extracted data locally.")
        archive_path = PROJECT_ROOT / cfg.get("archive_rel", "")
        if archive_path.exists():
            print(f"  {ds_name}: extracting from archive {archive_path}...")
            extract_path.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_path)
            print(f"  {ds_name}: extracted to {extract_path}")
        else:
            print(f"  {ds_name}: no archive found at {archive_path}.")
            print(f"    Run notebook 01 first or ensure Drive/archives/ has the ZIP files.")
    else:
        print(f"  {ds_name}: extracted data already present at {extract_path}")

print("--- Restore complete ---\n")

Manifests dir:  /content/CausalMask-XAI/data/manifests
Splits dir:     /content/CausalMask-XAI/data/splits
Reports dir:    /content/CausalMask-XAI/reports
Phases dir:     /content/CausalMask-XAI/artifacts/phases
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Artifacts will sync to /content/drive/MyDrive/CausalMask-XAI

--- Restoring Phase 2 artifacts from Drive ---
  busi: extracted data already present at /content/CausalMask-XAI/data/raw/extracted/busi
  bus_uclm: extracted data already present at /content/CausalMask-XAI/data/raw/extracted/bus_uclm
--- Restore complete ---



## 3.5 — Detect datasets on disk

Check whether extracted data exists (from Drive restore or local).
If not, attempt archive extraction. This is needed because Phase 3
reads actual image files for pHash and SSIM computation.

In [7]:
def detect_dataset(busi_cfg, uclm_cfg) -> dict:
    status = {"busi": {"status": "missing"}, "bus_uclm": {"status": "missing"}}

    busi_extract = PROJECT_ROOT / busi_cfg["extract_rel"]
    uclm_extract = PROJECT_ROOT / uclm_cfg["extract_rel"]
    busi_archive = PROJECT_ROOT / busi_cfg["archive_rel"]
    uclm_archive = PROJECT_ROOT / uclm_cfg["archive_rel"]

    # BUSI
    if busi_extract.exists() and any(busi_extract.iterdir()):
        status["busi"] = {"status": "extracted", "path": str(busi_extract)}
        print(f"BUSI: extracted data found at {busi_extract}")
    elif busi_archive.exists():
        print(f"BUSI: archive found at {busi_archive}, extracting...")
        busi_extract.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(busi_archive, "r") as zf:
            zf.extractall(busi_extract)
        status["busi"] = {"status": "extracted", "path": str(busi_extract)}
        print(f"BUSI: extracted to {busi_extract}")
    else:
        print(f"BUSI: NOT FOUND at {busi_extract} or {busi_archive}")
        print("  Run notebooks/01_download_and_extract_datasets.ipynb first or ensure Drive has extracted data.")

    # BUS-UCLM
    if uclm_extract.exists() and any(uclm_extract.iterdir()):
        status["bus_uclm"] = {"status": "extracted", "path": str(uclm_extract)}
        print(f"BUS-UCLM: extracted data found at {uclm_extract}")
    elif uclm_archive.exists():
        print(f"BUS-UCLM: archive found at {uclm_archive}, extracting...")
        uclm_extract.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(uclm_archive, "r") as zf:
            zf.extractall(uclm_extract)
        status["bus_uclm"] = {"status": "extracted", "path": str(uclm_extract)}
        print(f"BUS-UCLM: extracted to {uclm_extract}")
    else:
        print(f"BUS-UCLM: NOT FOUND at {uclm_extract} or {uclm_archive}")
        print("  Run notebooks/01_download_and_extract_datasets.ipynb first or ensure Drive has extracted data.")

    return status


busi_cfg = CONFIG["datasets"]["busi"]
uclm_cfg = CONFIG["datasets"]["bus_uclm"]

dataset_status = detect_dataset(busi_cfg, uclm_cfg)
print(f"\nDataset status: {json.dumps(dataset_status, indent=2)}")
missing = [k for k, v in dataset_status.items() if v["status"] != "extracted"]
if missing:
    print(f"\nWARNING: Missing datasets: {missing}. Phase 3 will fail if it needs actual image files.")


def check_manifest_available() -> bool:
    busi_manifest_path = MANIFESTS_DIR / f"busi_manifest_{CONFIG['manifest_version']}.parquet"
    return busi_manifest_path.exists()


if not check_manifest_available():
    print("\nERROR: BUSI manifest not found. Phase 2 must be completed first.")
    print("Make sure your Drive has manifests/ or run Phase 2 on this Colab instance.")

BUSI: extracted data found at /content/CausalMask-XAI/data/raw/extracted/busi
BUS-UCLM: extracted data found at /content/CausalMask-XAI/data/raw/extracted/bus_uclm

Dataset status: {
  "busi": {
    "status": "extracted",
    "path": "/content/CausalMask-XAI/data/raw/extracted/busi"
  },
  "bus_uclm": {
    "status": "extracted",
    "path": "/content/CausalMask-XAI/data/raw/extracted/bus_uclm"
  }
}


## 3.6 — Import Python modules

Import the reusable modules for duplicate detection and split generation.

In [8]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

import pandas as pd

from causalmask.data.manifest import (
    SampleRecord,
    validate_manifest,
    compute_sha256,
    compute_mask_area_fraction,
)
from causalmask.data.duplicate_audit import (
    find_exact_duplicates,
    compute_phashes_for_manifest,
    find_near_duplicate_candidates,
    verify_candidate_pairs,
    assign_duplicate_clusters,
    build_grouped_manifest,
    compute_image_similarity,
)
from causalmask.data.splits import (
    create_grouped_kfold_split,
    validate_split_disjointness,
    compute_split_digest,
    compute_manifest_digest,
    save_split,
)

print("All modules imported successfully.")

All modules imported successfully.


## 3.7 — Load Phase 2 BUSI manifest

Load the validated BUSI manifest from Phase 2.
Restores from Drive if missing locally.

In [9]:
busi_manifest_path = MANIFESTS_DIR / f"busi_manifest_{CONFIG['manifest_version']}.parquet"
print(f"Loading BUSI manifest: {busi_manifest_path}")

# Final fallback: try Drive one more time before failing
if not busi_manifest_path.exists():
    restored = restore_from_drive("manifests", busi_manifest_path.name, MANIFESTS_DIR)
    if not restored:
        raise FileNotFoundError(
            f"BUSI manifest not found at {busi_manifest_path}\n"
            "Run Phase 2 notebook first or ensure Drive has manifests/ with this file."
        )

df_busi = pd.read_parquet(busi_manifest_path)
print(f"BUSI manifest loaded: {len(df_busi)} samples, {len(df_busi.columns)} columns")
print(f"Columns: {list(df_busi.columns)}")
print(f"Label distribution: {df_busi['normalized_label'].value_counts().to_dict()}")

Loading BUSI manifest: /content/CausalMask-XAI/data/manifests/busi_manifest_v1.parquet
BUSI manifest loaded: 780 samples, 17 columns
Columns: ['sample_id', 'dataset', 'image_path', 'mask_path', 'raw_label', 'normalized_label', 'included_in_primary_task', 'patient_id', 'provisional_group_id', 'image_width', 'image_height', 'channels', 'image_sha256', 'mask_sha256', 'mask_area_fraction', 'has_mask', 'quality_flags']
Label distribution: {'benign': 437, 'malignant': 210, 'normal': 133}


## 3.8 — Detect exact duplicates (SHA-256)

Find images with identical SHA-256 hashes. These are exact pixel-for-pixel duplicates.

In [10]:
exact_dups = find_exact_duplicates(df_busi, PROJECT_ROOT)

if exact_dups.empty:
    print("No exact duplicates found.")
else:
    print(f"Exact duplicate groups: {len(exact_dups)}")
    print(f"Total duplicate images: {exact_dups['cluster_size'].sum()}")
    for _, row in exact_dups.iterrows():
        print(f"  {row['cluster_id']}: {row['sample_ids']}")

Exact duplicate groups: 1
Total duplicate images: 2
  exact_1: ['busi_benign_benign__433_', 'busi_malignant_malignant__145_']


## 3.9 — Compute perceptual hashes

Compute DCT-based pHash for every BUSI image. Used to find near-duplicate candidates.

In [11]:
def progress_callback(current: int, total: int):
    print(f"  pHash progress: {current}/{total}")

print(f"Computing pHashes for {len(df_busi)} BUSI images...")
phash_df = compute_phashes_for_manifest(
    df_busi,
    PROJECT_ROOT,
    hash_size=CONFIG["phash_hash_size"],
    highfreq_factor=CONFIG["phash_highfreq_factor"],
    batch_callback=progress_callback,
)

n_valid = phash_df["phash"].notna().sum()
n_failed = phash_df["phash"].isna().sum()
print(f"pHashes computed: {n_valid} success, {n_failed} failed")
print(f"Example pHash: {phash_df['phash'].iloc[0][:32]}...")

Computing pHashes for 780 BUSI images...
  pHash progress: 78/780
  pHash progress: 156/780
  pHash progress: 234/780
  pHash progress: 312/780
  pHash progress: 390/780
  pHash progress: 468/780
  pHash progress: 546/780
  pHash progress: 624/780
  pHash progress: 702/780
  pHash progress: 780/780
pHashes computed: 780 success, 0 failed
Example pHash: d7d24b88629e3335...


## 3.10 — Find near-duplicate candidates

Pairwise comparison of perceptual hashes to identify candidate near-duplicates.
Candidates are pairs with similarity >= min_similarity and < 1.0.

In [12]:
from causalmask.data.duplicate_audit import find_near_duplicate_candidates

candidates = find_near_duplicate_candidates(
    phash_df,
    min_similarity=CONFIG["phash_min_similarity"],
)

print(f"Candidate pairs found: {len(candidates)}")
if not candidates.empty:
    print(f"Similarity range: {candidates['phash_similarity'].min():.4f} - {candidates['phash_similarity'].max():.4f}")
    print("\nTop 10 candidates by similarity:")
    top = candidates.sort_values("phash_similarity", ascending=False).head(10)
    for _, row in top.iterrows():
        print(f"  {row['sample_id_a']} <-> {row['sample_id_b']}: sim={row['phash_similarity']:.4f}")

Candidate pairs found: 544
Similarity range: 0.7500 - 0.9688

Top 10 candidates by similarity:
  busi_normal_normal__43_ <-> busi_normal_normal__50_: sim=0.9688
  busi_normal_normal__42_ <-> busi_normal_normal__49_: sim=0.9688
  busi_normal_normal__42_ <-> busi_normal_normal__62_: sim=0.9688
  busi_normal_normal__42_ <-> busi_normal_normal__45_: sim=0.9688
  busi_normal_normal__41_ <-> busi_normal_normal__47_: sim=0.9688
  busi_normal_normal__38_ <-> busi_normal_normal__53_: sim=0.9688
  busi_normal_normal__41_ <-> busi_normal_normal__44_: sim=0.9688
  busi_normal_normal__40_ <-> busi_normal_normal__59_: sim=0.9688
  busi_normal_normal__40_ <-> busi_normal_normal__46_: sim=0.9688
  busi_normal_normal__131_ <-> busi_normal_normal__19_: sim=0.9688


## 3.11 — Verify near-duplicate candidates with SSIM

Verify candidate pairs using SSIM (structural similarity) at 256x256 resolution.
A candidate is confirmed when SSIM >= threshold.

In [13]:
if not candidates.empty:
    print(f"Verifying {len(candidates)} candidate pairs with SSIM...")
    verified = verify_candidate_pairs(
        candidates,
        df_busi,
        PROJECT_ROOT,
        ssim_threshold=CONFIG["ssim_verification_threshold"],
    )
    n_verified = verified["is_verified_near_duplicate"].sum() if not verified.empty else 0
    print(f"Verified near-duplicates: {n_verified} / {len(verified)} candidate pairs")
    
    if n_verified > 0:
        print("\nVerified pairs:")
        verified_pairs = verified[verified["is_verified_near_duplicate"]]
        for _, row in verified_pairs.iterrows():
            print(f"  {row['sample_id_a']} <-> {row['sample_id_b']}: SSIM={row['ssim']:.4f}")
else:
    verified = pd.DataFrame()
    print("No candidates to verify.")

# Save the verified dataframe
verified_out = MANIFESTS_DIR / f"duplicate_candidates_{CONFIG['duplicate_version']}.parquet"
verified.to_parquet(verified_out, index=False)
save_to_drive(verified_out, "manifests")
print(f"\nSaved candidate verification: {verified_out}")

Verifying 544 candidate pairs with SSIM...
Verified near-duplicates: 165 / 544 candidate pairs

Verified pairs:
  busi_benign_benign__1_ <-> busi_benign_benign__318_: SSIM=0.9273
  busi_benign_benign__108_ <-> busi_benign_benign__114_: SSIM=0.9106
  busi_benign_benign__108_ <-> busi_benign_benign__116_: SSIM=0.8583
  busi_benign_benign__108_ <-> busi_benign_benign__94_: SSIM=0.9316
  busi_benign_benign__110_ <-> busi_benign_benign__158_: SSIM=0.9318
  busi_benign_benign__114_ <-> busi_benign_benign__116_: SSIM=0.9027
  busi_benign_benign__114_ <-> busi_benign_benign__119_: SSIM=0.8594
  busi_benign_benign__114_ <-> busi_benign_benign__94_: SSIM=0.8668
  busi_benign_benign__116_ <-> busi_benign_benign__119_: SSIM=0.9383
  busi_benign_benign__116_ <-> busi_benign_benign__122_: SSIM=0.8913
  busi_benign_benign__119_ <-> busi_benign_benign__122_: SSIM=0.9376
  busi_benign_benign__12_ <-> busi_benign_benign__329_: SSIM=0.9238
  busi_benign_benign__128_ <-> busi_benign_benign__30_: SSIM=0.94

## 3.12 — Assign duplicate clusters

Use Union-Find to assign every sample to a cluster:
- Exact duplicates share a cluster
- Verified near-duplicates share a cluster (transitive closure)
- Singletons are their own cluster

In [14]:
clusters = assign_duplicate_clusters(exact_dups, verified, df_busi)

n_near = clusters["is_near_duplicate"].sum()
n_exact = clusters["is_exact_duplicate"].sum()
n_clusters = clusters["group_id"].nunique()
n_singletons = (clusters["cluster_size"] == 1).sum()

print(f"Total clusters: {n_clusters}")
print(f"  Exact duplicates: {n_exact} images")
print(f"  Near-duplicates:  {n_near} images")
print(f"  Singletons:       {n_singletons} groups")
print(f"  Multi-sample clusters: ", end="")
multi = clusters[clusters["cluster_size"] > 1]["group_id"].nunique()
print(f"{multi} clusters have size >= 2")

if multi > 0:
    print("\nMulti-sample clusters:")
    for cid in clusters[clusters["cluster_size"] > 1]["group_id"].unique():
        members = clusters[clusters["group_id"] == cid]["sample_id"].tolist()
        print(f"  {cid}: {members}")

# Save clusters
clusters_out = MANIFESTS_DIR / f"duplicate_clusters_{CONFIG['duplicate_version']}.parquet"
clusters.to_parquet(clusters_out, index=False)
save_to_drive(clusters_out, "manifests")
print(f"\nSaved clusters: {clusters_out}")

Total clusters: 643
  Exact duplicates: 2 images
  Near-duplicates:  237 images
  Singletons:       541 groups
  Multi-sample clusters: 102 clusters have size >= 2

Multi-sample clusters:
  near_1: ['busi_benign_benign__1_', 'busi_benign_benign__318_']
  near_2: ['busi_benign_benign__108_', 'busi_benign_benign__114_', 'busi_benign_benign__116_', 'busi_benign_benign__119_', 'busi_benign_benign__122_', 'busi_benign_benign__61_', 'busi_benign_benign__94_']
  near_3: ['busi_benign_benign__110_', 'busi_benign_benign__158_']
  near_4: ['busi_benign_benign__12_', 'busi_benign_benign__329_']
  near_5: ['busi_benign_benign__128_', 'busi_benign_benign__30_']
  near_6: ['busi_benign_benign__129_', 'busi_benign_benign__44_']
  near_7: ['busi_benign_benign__13_', 'busi_benign_benign__330_']
  near_8: ['busi_benign_benign__130_', 'busi_benign_benign__33_']
  near_9: ['busi_benign_benign__131_', 'busi_benign_benign__42_', 'busi_malignant_malignant__51_']
  near_10: ['busi_benign_benign__132_', 'busi_

## 3.13 — Build grouped manifest (v2)

Merge duplicate cluster assignments into the manifest to create the grouped v2 manifest.

In [15]:
df_busi_grouped = build_grouped_manifest(df_busi, clusters)

print(f"Grouped manifest: {len(df_busi_grouped)} samples")
print(f"New columns: group_id, near_duplicate_cluster, exact_duplicate_cluster")
print(f"Sample group_ids: {df_busi_grouped['group_id'].nunique()} unique")

# Verify all primary-task samples have a group_id
primary = df_busi_grouped[df_busi_grouped["included_in_primary_task"]]
missing_group = primary["group_id"].isna().sum()
print(f"Primary-task samples without group_id: {missing_group}")

# Save grouped manifest
grouped_path = MANIFESTS_DIR / f"busi_manifest_{CONFIG['grouped_manifest_version']}_grouped.parquet"
df_busi_grouped.to_parquet(grouped_path, index=False)
save_to_drive(grouped_path, "manifests")
print(f"\nSaved grouped manifest: {grouped_path}")

# Compute and record manifest digest
manifest_digest = compute_manifest_digest(df_busi_grouped)
print(f"Manifest digest (v2 grouped): {manifest_digest}")

Grouped manifest: 780 samples
New columns: group_id, near_duplicate_cluster, exact_duplicate_cluster
Sample group_ids: 643 unique
Primary-task samples without group_id: 0
  -> Synced to Drive: /content/drive/MyDrive/CausalMask-XAI/manifests/busi_manifest_v2_grouped.parquet

Saved grouped manifest: /content/CausalMask-XAI/data/manifests/busi_manifest_v2_grouped.parquet
Manifest digest (v2 grouped): 6462d283b3fcfe6657ece48daa8b7a0b09dc786bbfb34ed990c7d4d904f84304


## 3.14 — Create five-fold grouped split

Create one immutable five-fold split from the grouped manifest.
Requirements:
- Group-disjoint folds
- Approximate class stratification
- Validation from non-test development portion only
- Every primary-task sample in exactly one test fold
- No BUS-UCLM samples

In [16]:
split = create_grouped_kfold_split(
    df_busi_grouped,
    n_splits=CONFIG["n_splits"],
    seed=SEED,
    group_col="group_id",
    label_col="normalized_label",
    sample_id_col="sample_id",
    dataset_col="dataset",
    external_datasets=CONFIG["external_datasets"],
    valid_labels=CONFIG["binary_classification"],
)

split_digest = compute_split_digest(split)
print(f"Split digest: {split_digest}")

print(f"\nSplit statistics:")
stats = split["statistics"]
print(f"  Total internal samples: {stats['aggregate']['total_unique_samples']}")
print(f"  Total groups:           {stats['aggregate']['total_groups']}")
print(f"  Number of folds:        {stats['aggregate']['n_folds']}")

for fold_name, fold_stats in stats["per_fold"].items():
    parts = []
    for partition in ["train", "validation", "test"]:
        ps = fold_stats[partition]
        parts.append(f"{partition}={ps['n_samples']}({ps['n_groups']}g)")
    print(f"  {fold_name}: {', '.join(parts)}")

Split digest: 2a88e7ada1aff73e245d6d8b48693aaebb45ce5ad7568f6753f25cce4935f151

Split statistics:
  Total internal samples: 647
  Total groups:           542
  Number of folds:        5
  fold_0: train=443(359g), validation=103(89g), test=101(94g)
  fold_1: train=396(340g), validation=104(84g), test=147(118g)
  fold_2: train=439(359g), validation=104(89g), test=104(94g)
  fold_3: train=406(340g), validation=99(84g), test=142(118g)
  fold_4: train=400(340g), validation=94(84g), test=153(118g)


## 3.15 — Validate split disjointness

Hard assertions:
1. Sample disjointness across train/val/test
2. Group disjointness across partitions
3. Exact duplicate cluster disjointness
4. Near-duplicate cluster disjointness
5. Complete fold coverage
6. No external samples

In [17]:
validation = validate_split_disjointness(
    split,
    df_busi_grouped,
    group_col="group_id",
    sample_id_col="sample_id",
    dataset_col="dataset",
    external_datasets=CONFIG["external_datasets"],
)

print("=== Split Validation Results ===")
for check_name, passed in validation["checks"].items():
    icon = "PASS" if passed else "FAIL"
    print(f"  {icon}: {check_name}")

if validation["failures"]:
    print(f"\nFailures ({len(validation['failures'])}):")
    for f in validation["failures"]:
        print(f"  - {f}")

print(f"\nOverall: {'PASSED' if validation['passed'] else 'FAILED'}")

if not validation["passed"]:
    raise RuntimeError("Split validation failed. Cannot proceed until split is fixed.")

=== Split Validation Results ===
  PASS: sample_disjointness
  PASS: group_disjointness
  PASS: exact_duplicate_disjointness
  PASS: near_duplicate_disjointness
  PASS: complete_coverage
  PASS: no_external_contamination

Overall: PASSED


## 3.16 — Save split to JSON

Save with embedded digest. This split is immutable — any correction requires v2.

In [18]:
split_path = SPLITS_DIR / f"{CONFIG['split_name']}.json"
save_split(split, split_path)
save_to_drive(split_path, "splits")
print(f"\nSplit saved: {split_path}")
print(f"  digest: {split['metadata'].get('split_digest', 'unknown')[:32]}...")

  -> Synced to Drive: /content/drive/MyDrive/CausalMask-XAI/splits/busi_binary_grouped_5fold_v1.json

Split saved: /content/CausalMask-XAI/data/splits/busi_binary_grouped_5fold_v1.json
  digest: 2a88e7ada1aff73e245d6d8b48693aae...


## 3.17 — Split reproducibility test

Verify that the split is reproducible from the same seed and configuration.

In [19]:
# Regenerate from same seed and verify digest matches
split_regen = create_grouped_kfold_split(
    df_busi_grouped,
    n_splits=CONFIG["n_splits"],
    seed=SEED,
    group_col="group_id",
    label_col="normalized_label",
    sample_id_col="sample_id",
    dataset_col="dataset",
    external_datasets=CONFIG["external_datasets"],
    valid_labels=CONFIG["binary_classification"],
)

regen_digest = compute_split_digest(split_regen)
original_digest = split["metadata"].get("split_digest", compute_split_digest(split))

if regen_digest == original_digest:
    print(f"PASS: Split reproducibility verified (digest: {regen_digest[:16]}...)")
else:
    print(f"FAIL: Digest mismatch!")
    print(f"  Original: {original_digest}")
    print(f"  Regenerated: {regen_digest}")

PASS: Split reproducibility verified (digest: 2a88e7ada1aff73e...)


## 3.18 — Generate duplicate & split audit report

In [20]:
def write_duplicate_and_split_audit():
    lines = [
        "# Duplicate Detection and Split Audit — Phase 3",
        "",
        f"**Generated:** {datetime.now(timezone.utc).isoformat()}",
        f"**Project root:** `{PROJECT_ROOT}`",
        f"**Seed:** {SEED}",
        "",
        "---",
        "",
        "## Grouping Semantics",
        "",
        "BUSI does not provide reliable patient identifiers. Grouping is based on:",
        "- **Exact duplicates** (SHA-256 identity)",
        "- **Near-duplicates** (pHash similarity + SSIM verification)",
        "- All other samples are singletons",
        "",
        "This is **duplicate-group splitting**, not patient-level splitting.",
        "This limitation is documented because BUSI filenames do not encode",
        "consistent patient identifiers.",
        "",
        "## Duplicate Summary",
        "",
        f"- **Exact duplicate groups:** {len(exact_dups)}",
        f"- **Near-duplicate groups:** {clusters['is_near_duplicate'].sum()}",
        f"- **Total clusters:** {clusters['group_id'].nunique()}",
        f"- **Singletons:** {(clusters['cluster_size'] == 1).sum()}",
        "",
        "## Split Summary",
        "",
        f"- **Split file:** `{CONFIG['split_name']}.json`",
        f"- **Algorithm:** grouped_stratified_kfold",
        f"- **Seed:** {SEED}",
        f"- **Folds:** {CONFIG['n_splits']}",
        f"- **Total samples:** {stats['aggregate']['total_unique_samples']}",
        f"- **Total groups:** {stats['aggregate']['total_groups']}",
        f"- **Split digest:** `{split_digest}`",
        f"- **Manifest digest (v2 grouped):** `{manifest_digest}`",
        "",
        "### Per-fold counts",
        "",
        "| Fold | Train | Val | Test | Train groups | Val groups | Test groups |",
        "|------|-------|-----|------|-------------|-----------|------------|",
    ]
    
    for fold_name in sorted(stats["per_fold"].keys()):
        fs = stats["per_fold"][fold_name]
        row = (
            f"| {fold_name} "
            f"| {fs['train']['n_samples']} "
            f"| {fs['validation']['n_samples']} "
            f"| {fs['test']['n_samples']} "
            f"| {fs['train']['n_groups']} "
            f"| {fs['validation']['n_groups']} "
            f"| {fs['test']['n_groups']} |"
        )
        lines.append(row)
    
    lines.extend([
        "",
        "### Class counts by partition",
        "",
        "| Partition | Benign | Malignant |",
        "|-----------|--------|-----------|",
    ])
    
    for fold_name in sorted(stats["per_fold"].keys()):
        fs = stats["per_fold"][fold_name]
        for partition in ["train", "validation", "test"]:
            cc = fs[partition]["class_counts"]
            row = f"| {fold_name} {partition} | {cc.get('benign', 0)} | {cc.get('malignant', 0)} |"
            lines.append(row)
    
    lines.extend([
        "",
        "## Validation Results",
        "",
    ])
    
    for check_name, passed in validation["checks"].items():
        status = "PASS" if passed else "FAIL"
        lines.append(f"- **{status}:** {check_name}")
    
    if validation["failures"]:
        lines.append("")
        lines.append("### Failures")
        for f in validation["failures"]:
            lines.append(f"- {f}")
    
    lines.extend([
        "",
        "---",
        "",
        "## Duplicate cluster assignments",
        "",
    ])
    
    multi_clusters = clusters[clusters["cluster_size"] > 1]
    if not multi_clusters.empty:
        lines.append("| Cluster ID | Members | Size | Type |")
        lines.append("|-----------|---------|------|------|")
        for _, row in multi_clusters.iterrows():
            ctype = "exact" if row["is_exact_duplicate"] else "near"
            lines.append(f"| {row['group_id']} | {row['sample_id']} | {row['cluster_size']} | {ctype} |")
    else:
        lines.append("No duplicate clusters found.")
    
    lines.append("")
    lines.append("## Immutability notice")
    lines.append("")
    lines.append(
        "This split is immutable. Any future correction must create v2 "
        "and record a deviation in reports/deviations.md."
    )
    
    return "\n".join(lines)


report_content = write_duplicate_and_split_audit()
report_path = REPORTS_DIR / "duplicate_and_split_audit.md"
with open(report_path, "w") as f:
    f.write(report_content)
save_to_drive(report_path, "reports")
print(f"Saved report: {report_path}")
print(report_content)

  -> Synced to Drive: /content/drive/MyDrive/CausalMask-XAI/reports/duplicate_and_split_audit.md
Saved report: /content/CausalMask-XAI/reports/duplicate_and_split_audit.md
# Duplicate Detection and Split Audit — Phase 3

**Generated:** 2026-07-22T07:21:10.909445+00:00
**Project root:** `/content/CausalMask-XAI`
**Seed:** 42

---

## Grouping Semantics

BUSI does not provide reliable patient identifiers. Grouping is based on:
- **Exact duplicates** (SHA-256 identity)
- **Near-duplicates** (pHash similarity + SSIM verification)
- All other samples are singletons

This is **duplicate-group splitting**, not patient-level splitting.
This limitation is documented because BUSI filenames do not encode
consistent patient identifiers.

## Duplicate Summary

- **Exact duplicate groups:** 1
- **Near-duplicate groups:** 237
- **Total clusters:** 643
- **Singletons:** 541

## Split Summary

- **Split file:** `busi_binary_grouped_5fold_v1.json`
- **Algorithm:** grouped_stratified_kfold
- **Seed:** 42

## 3.19 — Write phase status JSON

Machine-readable status for pipeline orchestration.

In [21]:
phase_status = {
    "phase": "03",
    "name": "Duplicate Audit and Fixed Splits",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config": CONFIG,
    "environment_summary": {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "duplicate_summary": {
        "exact_duplicate_groups": len(exact_dups),
        "near_duplicate_images": int(clusters["is_near_duplicate"].sum()),
        "total_clusters": int(clusters["group_id"].nunique()),
        "singletons": int((clusters["cluster_size"] == 1).sum()),
        "multi_sample_clusters": int(
            clusters[clusters["cluster_size"] > 1]["group_id"].nunique()
        ),
    },
    "split_summary": {
        "total_internal_samples": stats["aggregate"]["total_unique_samples"],
        "total_groups": stats["aggregate"]["total_groups"],
        "n_folds": stats["aggregate"]["n_folds"],
        "split_digest": split_digest,
        "manifest_digest": manifest_digest,
        "split_file": str(split_path),
        "split_name": CONFIG["split_name"],
    },
    "split_validation": {
        "passed": validation["passed"],
        "checks": validation["checks"],
        "failures": validation["failures"],
    },
    "grouping_semantics": (
        "BUSI does not provide reliable patient identifiers. "
        "Grouping is based on exact SHA-256 duplicates and "
        "verified near-duplicate clusters (pHash + SSIM). "
        "This is duplicate-group splitting, not patient-level splitting."
    ),
    "files_created": [
        "notebooks/03_duplicate_audit_and_fixed_splits.ipynb",
        "src/causalmask/data/duplicate_audit.py",
        "src/causalmask/data/splits.py",
        "tests/unit/test_duplicate_audit.py",
        "tests/unit/test_splits.py",
    ],
    "generated_artifacts": [
        str(grouped_path),
        str(verified_out),
        str(clusters_out),
        str(split_path),
        str(report_path),
    ],
    "completed_checks": [
        "manifest_loaded",
        "exact_duplicates_detected",
        "perceptual_hashes_computed",
        "near_duplicate_candidates_found",
        "candidates_verified_with_ssim",
        "duplicate_clusters_assigned",
        "grouped_manifest_saved",
        "five_fold_split_created",
        "split_reproducibility_verified",
        "split_validation_passed",
        "split_saved_with_digest",
        "audit_report_generated",
    ],
    "failed_checks": [],
    "status_label": "executed",
    "phase_state": "phase_3_gate_passed" if validation["passed"] else "phase_3_gate_failed",
}

status_path = PHASES_DIR / "phase_03_status.json"
with open(status_path, "w") as f:
    json.dump(phase_status, f, indent=2)
save_to_drive(status_path, "artifacts")

print(f"Phase status saved: {status_path}")
print(json.dumps(phase_status, indent=2))

  -> Synced to Drive: /content/drive/MyDrive/CausalMask-XAI/artifacts/phase_03_status.json
Phase status saved: /content/CausalMask-XAI/artifacts/phases/phase_03_status.json
{
  "phase": "03",
  "name": "Duplicate Audit and Fixed Splits",
  "timestamp_utc": "2026-07-22T07:21:22.338471+00:00",
  "project_root": "/content/CausalMask-XAI",
  "config": {
    "project_root": "/content/CausalMask-XAI",
    "python": "3.12.13",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
    "torch": "2.11.0+cpu",
    "cuda_available": false,
    "cuda_version": null,
    "gpu_name": null,
    "phase": "03",
    "phase_name": "Duplicate Audit and Fixed Splits",
    "timestamp_utc": "2026-07-22T07:19:36.622132+00:00",
    "datasets": {
      "busi": {
        "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
        "extract_rel": "data/raw/extracted/busi",
        "manifest_rel": "data/manifests/busi_manifest_v1.parquet"
      },
      "bus_uclm": {
        "archive_rel": "da

## 3.20 — Phase 3 gate check

Gate criteria:
- Grouping semantics are documented honestly
- Duplicate candidates and clusters are saved
- Five-fold split is fixed
- All disjointness tests pass
- Split and manifest digests are recorded
- BUS-UCLM remains external-only

In [22]:
def check_phase3_gate() -> dict:
    results = {}
    
    # 1. Grouping semantics documented
    results["grouping_semantics_documented"] = (
        "duplicate-group splitting" in phase_status["grouping_semantics"].lower()
    )
    
    # 2. Duplicate candidates and clusters saved
    results["duplicate_candidates_saved"] = verified_out.exists()
    results["duplicate_clusters_saved"] = clusters_out.exists()
    
    # 3. Five-fold split is fixed
    results["split_file_exists"] = split_path.exists()
    
    # 4. All disjointness tests pass
    results["sample_disjointness"] = validation["checks"].get("sample_disjointness", False)
    results["group_disjointness"] = validation["checks"].get("group_disjointness", False)
    results["exact_duplicate_disjointness"] = validation["checks"].get(
        "exact_duplicate_disjointness", False
    )
    results["near_duplicate_disjointness"] = validation["checks"].get(
        "near_duplicate_disjointness", False
    )
    results["complete_coverage"] = validation["checks"].get("complete_coverage", False)
    
    # 5. Split and manifest digests recorded
    results["split_digest_recorded"] = bool(split_digest)
    results["manifest_digest_recorded"] = bool(manifest_digest)
    
    # 6. BUS-UCLM remains external-only
    results["no_external_contamination"] = validation["checks"].get(
        "no_external_contamination", False
    )
    
    # 7. Seed and algorithm recorded
    results["seed_recorded"] = split["metadata"].get("seed") == SEED
    results["algorithm_recorded"] = bool(split["metadata"].get("algorithm"))
    
    return results


gate_results = check_phase3_gate()

all_pass = all(gate_results.values())

for k, v in gate_results.items():
    icon = "PASS" if v else "FAIL"
    print(f"  {icon}: {k}")

print(f"\n{'='*60}")
if all_pass:
    print("PHASE 3 GATE: PASSED")
else:
    failed = [k for k, v in gate_results.items() if not v]
    print(f"PHASE 3 GATE: FAILED — {failed}")
print(f"{'='*60}")

  PASS: grouping_semantics_documented
  PASS: duplicate_candidates_saved
  PASS: duplicate_clusters_saved
  PASS: split_file_exists
  PASS: sample_disjointness
  PASS: group_disjointness
  PASS: exact_duplicate_disjointness
  PASS: near_duplicate_disjointness
  PASS: complete_coverage
  PASS: split_digest_recorded
  PASS: manifest_digest_recorded
  PASS: no_external_contamination
  PASS: seed_recorded
  PASS: algorithm_recorded

PHASE 3 GATE: PASSED


## 3.21 — Summary

Reports:
1. Current evidence state
2. Files created or changed
3. Notebook cells or commands actually executed
4. Tests passed and failed
5. Generated artifacts
6. Unresolved risks or deviations
7. Whether the phase gate passed

In [23]:
summary = {
    "current_evidence_state": (
        "Duplicate relationships detected and documented. "
        "One immutable five-fold grouped split created."
        if all_pass
        else "Split generation failed — see gate check for details."
    ),
    "files_created": phase_status["files_created"],
    "notebook_cells_executed": "All cells in 03_duplicate_audit_and_fixed_splits.ipynb",
    "tests_passed": [k for k, v in gate_results.items() if v],
    "tests_failed": [k for k, v in gate_results.items() if not v],
    "generated_artifacts": phase_status["generated_artifacts"],
    "unresolved_risks": (
        ["Grouping is duplicate-based, not patient-level. See grouping_semantics."]
    ),
    "deviations": [],
    "phase_gate_passed": all_pass,
    "next_action": (
        "Proceed to Phase 4 (baseline pipeline) after confirming split and manifests."
        if all_pass
        else "Fix split validation issues and re-run this notebook."
    ),
}

print(json.dumps(summary, indent=2))
print(f"\n{'='*60}")
print(f"PHASE 03 GATE: {'PASSED' if all_pass else 'FAILED'}")
print(f"{'='*60}")

{
  "current_evidence_state": "Duplicate relationships detected and documented. One immutable five-fold grouped split created.",
  "files_created": [
    "notebooks/03_duplicate_audit_and_fixed_splits.ipynb",
    "src/causalmask/data/duplicate_audit.py",
    "src/causalmask/data/splits.py",
    "tests/unit/test_duplicate_audit.py",
    "tests/unit/test_splits.py"
  ],
  "notebook_cells_executed": "All cells in 03_duplicate_audit_and_fixed_splits.ipynb",
  "tests_passed": [
    "grouping_semantics_documented",
    "duplicate_candidates_saved",
    "duplicate_clusters_saved",
    "split_file_exists",
    "sample_disjointness",
    "group_disjointness",
    "exact_duplicate_disjointness",
    "near_duplicate_disjointness",
    "complete_coverage",
    "split_digest_recorded",
    "manifest_digest_recorded",
    "no_external_contamination",
    "seed_recorded",
    "algorithm_recorded"
  ],
  "tests_failed": [],
  "generated_artifacts": [
    "/content/CausalMask-XAI/data/manifests/busi_ma

## 3.22 — Drive save verification

All artifacts were saved to Drive progressively during the pipeline. This cell verifies that Drive has all expected files.

In [24]:
if DRIVE_BASE is not None:
    print(f"Drive artifacts at: {DRIVE_BASE}")
    for subdir in ["manifests", "splits", "reports", "artifacts"]:
        p = DRIVE_BASE / subdir
        if p.exists():
            files = list(p.iterdir())
            print(f"  {subdir}/: {len(files)} file(s)")
            for f in files:
                print(f"    {f.name}")
        else:
            print(f"  {subdir}/: (empty)")
    print("\nDrive save verified.")
else:
    print("Drive not mounted — progressive saves skipped. Artifacts exist locally.")

Drive artifacts at: /content/drive/MyDrive/CausalMask-XAI
  manifests/: 7 file(s)
    busi_manifest_v1.parquet
    bus_uclm_manifest_v1.parquet
    bus_uclm_manifest_summary_v1.json
    busi_manifest_summary_v1.json
    duplicate_candidates_v1.parquet
    busi_manifest_v2_grouped.parquet
    duplicate_clusters_v1.parquet
  splits/: 1 file(s)
    busi_binary_grouped_5fold_v1.json
  reports/: 2 file(s)
    data_audit.md
    duplicate_and_split_audit.md
  artifacts/: 2 file(s)
    phase_02_status.json
    phase_03_status.json

Drive save verified.


---

**Phase 3 complete.** Do NOT start Phase 4 automatically.